# Evaluation of Speaker-Phrase Pair Extraction

This notebook evaluates the quality of extracted speaker-phrase pairs for opening and closing expressions.

## Metrics:
1. **Jaccard Index** - Set similarity between gold and predicted pairs
2. **Precision, Recall, F1** - Standard classification metrics
3. **Greedy Aligned Phrase Match** - Sort lexicographically, align greedily, evaluate phrase overlap

In [ ]:
import pandas as pd
import numpy as np
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

: 

## Load Data

In [ ]:
# Load gold and extracted pairs
gold_df = pd.read_excel('..\data\raw\gold\gold.xlsx')
extracted_df = pd.read_excel('..\artifacts\test_piter_fm\detailed_pairs.xlsx')

print("Gold pairs:")
display(gold_df)
print("\nExtracted pairs:")
display(extracted_df)

## Preprocessing

Convert all pairs to lowercase and parse into (speaker, phrase) tuples.

In [ ]:
def parse_pairs(df, column, category):
    """
    Parse pairs from a column, lowercase, and return as set of (speaker, phrase, category) tuples.
    Handles multiple pairs per cell separated by ';'.
    """
    pairs = set()
    for val in df[column]:
        if pd.isna(val):
            continue
        val = str(val).strip()
        if val == '' or val == '0':
            continue
        
        # Split by ';' to handle multiple pairs per cell
        for pair_str in val.split(';'):
            pair_str = pair_str.strip().lower()
            if pair_str == '':
                continue
            if ' - ' in pair_str:
                parts = pair_str.split(' - ', 1)
                if len(parts) == 2:
                    speaker, phrase = parts[0].strip(), parts[1].strip()
                    pairs.add((speaker, phrase, category))
    return pairs

# Parse all categories (with category label included)
gold_opening = parse_pairs(gold_df, 'opening', 'opening')
gold_closing = parse_pairs(gold_df, 'closing', 'closing')
gold_all = gold_opening | gold_closing

pred_opening = parse_pairs(extracted_df, 'opening', 'opening')
pred_closing = parse_pairs(extracted_df, 'closing', 'closing')
pred_all = pred_opening | pred_closing

# Also create sets without category for per-category evaluation (speaker, phrase only)
def strip_category(pair_set):
    return {(speaker, phrase) for speaker, phrase, _ in pair_set}

gold_opening_simple = strip_category(gold_opening)
gold_closing_simple = strip_category(gold_closing)
pred_opening_simple = strip_category(pred_opening)
pred_closing_simple = strip_category(pred_closing)

print(f"Gold opening pairs: {len(gold_opening)}")
print(f"Gold closing pairs: {len(gold_closing)}")
print(f"Gold total pairs: {len(gold_all)}")
print()
print(f"Predicted opening pairs: {len(pred_opening)}")
print(f"Predicted closing pairs: {len(pred_closing)}")
print(f"Predicted total pairs: {len(pred_all)}")

## Evaluation Functions

In [ ]:
def jaccard_index(set1, set2):
    """Calculate Jaccard index between two sets."""
    if len(set1) == 0 and len(set2) == 0:
        return 1.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0.0

def precision_recall_f1(gold_set, pred_set):
    """Calculate precision, recall, and F1 score."""
    if len(pred_set) == 0:
        precision = 0.0
    else:
        precision = len(gold_set & pred_set) / len(pred_set)
    
    if len(gold_set) == 0:
        recall = 0.0
    else:
        recall = len(gold_set & pred_set) / len(gold_set)
    
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    
    return precision, recall, f1

def phrase_similarity(phrase1, phrase2):
    """Calculate similarity between two phrases using SequenceMatcher."""
    return SequenceMatcher(None, phrase1, phrase2).ratio()

def longest_common_prefix(s1, s2):
    """Calculate longest common prefix length."""
    min_len = min(len(s1), len(s2))
    for i in range(min_len):
        if s1[i] != s2[i]:
            return i
    return min_len

def combined_similarity(phrase1, phrase2, prefix_weight=0.3, seq_weight=0.7):
    """
    Combined similarity score using:
    - Longest common prefix (normalized)
    - SequenceMatcher ratio
    """
    if not phrase1 or not phrase2:
        return 0.0
    
    # Sequence similarity
    seq_sim = SequenceMatcher(None, phrase1, phrase2).ratio()
    
    # Prefix similarity (normalized by max length)
    lcp = longest_common_prefix(phrase1, phrase2)
    prefix_sim = lcp / max(len(phrase1), len(phrase2))
    
    return prefix_weight * prefix_sim + seq_weight * seq_sim

def greedy_alignment_evaluation(gold_set, pred_set, include_category=False, min_similarity=0.0):
    """
    Compute all valid pair similarities (matching speaker and optionally category),
    sort by similarity, and greedily assign best matches first.
    
    Args:
        gold_set: set of tuples - either (speaker, phrase) or (speaker, phrase, category)
        pred_set: set of tuples - either (speaker, phrase) or (speaker, phrase, category)
        include_category: if True, match on both speaker AND category
        min_similarity: minimum phrase similarity threshold for a valid match
    
    Returns:
        - exact_matches: count of exact phrase matches
        - avg_similarity: average phrase similarity for aligned pairs
        - alignment_details: list of alignment dictionaries
    """
    gold_list = list(gold_set)
    pred_list = list(pred_set)
    
    # Compute all valid (gold_idx, pred_idx, similarity) pairs
    candidate_pairs = []
    
    for g_idx, gold_item in enumerate(gold_list):
        if include_category:
            gold_speaker, gold_phrase, gold_category = gold_item
        else:
            gold_speaker, gold_phrase = gold_item
            gold_category = None
        
        for p_idx, pred_item in enumerate(pred_list):
            if include_category:
                pred_speaker, pred_phrase, pred_category = pred_item
            else:
                pred_speaker, pred_phrase = pred_item
                pred_category = None
            
            # Must match speaker
            if gold_speaker != pred_speaker:
                continue
            
            # If category is included, must also match category
            if include_category and gold_category != pred_category:
                continue
            
            # Compute combined similarity (prefix + sequence)
            sim = combined_similarity(gold_phrase, pred_phrase)
            
            if sim >= min_similarity:
                candidate_pairs.append((g_idx, p_idx, sim, gold_phrase, pred_phrase))
    
    # Sort by similarity (descending) - best matches first
    candidate_pairs.sort(key=lambda x: x[2], reverse=True)
    
    # Greedy assignment
    used_gold = set()
    used_pred = set()
    alignments = []
    matched_info = {}  # g_idx -> (p_idx, similarity, pred_phrase)
    
    for g_idx, p_idx, sim, gold_phrase, pred_phrase in candidate_pairs:
        if g_idx in used_gold or p_idx in used_pred:
            continue
        used_gold.add(g_idx)
        used_pred.add(p_idx)
        matched_info[g_idx] = (p_idx, sim, pred_phrase)
    
    # Build alignment results
    for g_idx, gold_item in enumerate(gold_list):
        if include_category:
            gold_speaker, gold_phrase, gold_category = gold_item
        else:
            gold_speaker, gold_phrase = gold_item
            gold_category = None
        
        if g_idx in matched_info:
            p_idx, sim, pred_phrase = matched_info[g_idx]
            if include_category:
                pred_speaker = pred_list[p_idx][0]
            else:
                pred_speaker = pred_list[p_idx][0]
            
            alignment = {
                'gold_speaker': gold_speaker,
                'gold_phrase': gold_phrase,
                'pred_speaker': pred_speaker,
                'pred_phrase': pred_phrase,
                'similarity': sim,
                'exact_match': gold_phrase == pred_phrase
            }
            if include_category:
                alignment['category'] = gold_category
            alignments.append(alignment)
        else:
            alignment = {
                'gold_speaker': gold_speaker,
                'gold_phrase': gold_phrase,
                'pred_speaker': None,
                'pred_phrase': None,
                'similarity': 0.0,
                'exact_match': False
            }
            if include_category:
                alignment['category'] = gold_category
            alignments.append(alignment)
    
    # Add unmatched predictions
    for p_idx, pred_item in enumerate(pred_list):
        if p_idx not in used_pred:
            if include_category:
                pred_speaker, pred_phrase, pred_category = pred_item
            else:
                pred_speaker, pred_phrase = pred_item
                pred_category = None
            
            alignment = {
                'gold_speaker': None,
                'gold_phrase': None,
                'pred_speaker': pred_speaker,
                'pred_phrase': pred_phrase,
                'similarity': 0.0,
                'exact_match': False
            }
            if include_category:
                alignment['category'] = pred_category
            alignments.append(alignment)
    
    # Calculate metrics
    matched = [a for a in alignments if a['pred_phrase'] is not None and a['gold_phrase'] is not None]
    exact_matches = sum(1 for a in matched if a['exact_match'])
    avg_similarity = np.mean([a['similarity'] for a in matched]) if matched else 0.0
    
    return {
        'exact_matches': exact_matches,
        'total_gold': len(gold_list),
        'total_pred': len(pred_list),
        'matched_pairs': len(matched),
        'avg_similarity': avg_similarity,
        'alignments': alignments
    }

## Evaluate All Pairs (Combined)

In [ ]:
print("=" * 60)
print("EVALUATION: ALL PAIRS (Opening + Closing Combined)")
print("=" * 60)

# Jaccard Index (on full tuples including category)
jaccard = jaccard_index(gold_all, pred_all)
print(f"\nJaccard Index: {jaccard:.4f}")

# Precision, Recall, F1 (exact match on speaker + phrase + category)
precision, recall, f1 = precision_recall_f1(gold_all, pred_all)
print("\nExact Match Metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# Greedy Alignment (matching on speaker AND category)
alignment_results = greedy_alignment_evaluation(gold_all, pred_all, include_category=True)
print("\nGreedy Alignment Evaluation (matching speaker + category):")
print(f"  Total Gold pairs:     {alignment_results['total_gold']}")
print(f"  Total Predicted:      {alignment_results['total_pred']}")
print(f"  Matched pairs:        {alignment_results['matched_pairs']}")
print(f"  Exact phrase matches: {alignment_results['exact_matches']}")
print(f"  Avg phrase similarity: {alignment_results['avg_similarity']:.4f}")

In [ ]:
# Show alignment details for all pairs
print("\nAlignment Details (All Pairs):")
print("-" * 100)
alignment_df = pd.DataFrame(alignment_results['alignments'])
display(alignment_df)

## Evaluate Opening Pairs

In [ ]:
print("=" * 60)
print("EVALUATION: OPENING PAIRS")
print("=" * 60)

# Jaccard Index
jaccard_open = jaccard_index(gold_opening_simple, pred_opening_simple)
print(f"\nJaccard Index: {jaccard_open:.4f}")

# Precision, Recall, F1
precision_open, recall_open, f1_open = precision_recall_f1(gold_opening_simple, pred_opening_simple)
print("\nExact Match Metrics:")
print(f"  Precision: {precision_open:.4f}")
print(f"  Recall:    {recall_open:.4f}")
print(f"  F1 Score:  {f1_open:.4f}")

# Greedy Alignment (matching on speaker only, within opening category)
alignment_open = greedy_alignment_evaluation(gold_opening_simple, pred_opening_simple, include_category=False)
print("\nGreedy Alignment Evaluation (matching speaker):")
print(f"  Total Gold pairs:     {alignment_open['total_gold']}")
print(f"  Total Predicted:      {alignment_open['total_pred']}")
print(f"  Matched pairs:        {alignment_open['matched_pairs']}")
print(f"  Exact phrase matches: {alignment_open['exact_matches']}")
print(f"  Avg phrase similarity: {alignment_open['avg_similarity']:.4f}")

In [ ]:
# Show alignment details for opening pairs
print("\nAlignment Details (Opening):")
alignment_open_df = pd.DataFrame(alignment_open['alignments'])
display(alignment_open_df)

## Evaluate Closing Pairs

In [ ]:
print("=" * 60)
print("EVALUATION: CLOSING PAIRS")
print("=" * 60)

# Jaccard Index
jaccard_close = jaccard_index(gold_closing_simple, pred_closing_simple)
print(f"\nJaccard Index: {jaccard_close:.4f}")

# Precision, Recall, F1
precision_close, recall_close, f1_close = precision_recall_f1(gold_closing_simple, pred_closing_simple)
print("\nExact Match Metrics:")
print(f"  Precision: {precision_close:.4f}")
print(f"  Recall:    {recall_close:.4f}")
print(f"  F1 Score:  {f1_close:.4f}")

# Greedy Alignment (matching on speaker only, within closing category)
alignment_close = greedy_alignment_evaluation(gold_closing_simple, pred_closing_simple, include_category=False)
print("\nGreedy Alignment Evaluation (matching speaker):")
print(f"  Total Gold pairs:     {alignment_close['total_gold']}")
print(f"  Total Predicted:      {alignment_close['total_pred']}")
print(f"  Matched pairs:        {alignment_close['matched_pairs']}")
print(f"  Exact phrase matches: {alignment_close['exact_matches']}")
print(f"  Avg phrase similarity: {alignment_close['avg_similarity']:.4f}")

In [ ]:
# Show alignment details for closing pairs
print("\nAlignment Details (Closing):")
alignment_close_df = pd.DataFrame(alignment_close['alignments'])
display(alignment_close_df)

## Summary Table

In [ ]:
# Create summary table
summary_data = {
    'Category': ['All Pairs', 'Opening', 'Closing'],
    'Gold Count': [len(gold_all), len(gold_opening_simple), len(gold_closing_simple)],
    'Pred Count': [len(pred_all), len(pred_opening_simple), len(pred_closing_simple)],
    'Jaccard': [jaccard, jaccard_open, jaccard_close],
    'Precision': [precision, precision_open, precision_close],
    'Recall': [recall, recall_open, recall_close],
    'F1': [f1, f1_open, f1_close],
    'Exact Matches': [
        alignment_results['exact_matches'],
        alignment_open['exact_matches'],
        alignment_close['exact_matches']
    ],
    'Avg Phrase Similarity': [
        alignment_results['avg_similarity'],
        alignment_open['avg_similarity'],
        alignment_close['avg_similarity']
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
display(summary_df.style.format({
    'Jaccard': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    'Avg Phrase Similarity': '{:.4f}'
}))

## Error Analysis

In [ ]:
print("\n" + "=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

# False Negatives (in gold but not in predictions)
false_negatives = gold_all - pred_all
print(f"\nFalse Negatives (missed pairs): {len(false_negatives)}")
for speaker, phrase, category in sorted(false_negatives):
    print(f"  - [{category}] {speaker}: {phrase}")

# False Positives (in predictions but not in gold)
false_positives = pred_all - gold_all
print(f"\nFalse Positives (extra pairs): {len(false_positives)}")
for speaker, phrase, category in sorted(false_positives):
    print(f"  - [{category}] {speaker}: {phrase}")

# True Positives (exact matches)
true_positives = gold_all & pred_all
print(f"\nTrue Positives (exact matches): {len(true_positives)}")
for speaker, phrase, category in sorted(true_positives):
    print(f"  - [{category}] {speaker}: {phrase}")

## Additional Metrics: Token-Level F1

In [ ]:
def token_level_f1(gold_phrase, pred_phrase):
    """Calculate token-level precision, recall, F1 for partial matching."""
    gold_tokens = set(gold_phrase.lower().split())
    pred_tokens = set(pred_phrase.lower().split())
    
    if len(pred_tokens) == 0:
        precision = 0.0
    else:
        precision = len(gold_tokens & pred_tokens) / len(pred_tokens)
    
    if len(gold_tokens) == 0:
        recall = 0.0
    else:
        recall = len(gold_tokens & pred_tokens) / len(gold_tokens)
    
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    
    return precision, recall, f1

# Calculate average token-level F1 across aligned pairs
all_alignments = alignment_results['alignments']
matched_alignments = [a for a in all_alignments 
                      if a['gold_phrase'] is not None and a['pred_phrase'] is not None]

if matched_alignments:
    token_scores = [token_level_f1(a['gold_phrase'], a['pred_phrase']) 
                    for a in matched_alignments]
    avg_token_precision = np.mean([s[0] for s in token_scores])
    avg_token_recall = np.mean([s[1] for s in token_scores])
    avg_token_f1 = np.mean([s[2] for s in token_scores])
    
    print("Token-Level Metrics (for aligned pairs):")
    print(f"  Avg Token Precision: {avg_token_precision:.4f}")
    print(f"  Avg Token Recall:    {avg_token_recall:.4f}")
    print(f"  Avg Token F1:        {avg_token_f1:.4f}")
else:
    print("No aligned pairs to evaluate token-level metrics.")